In [47]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

## Создание модели

In [48]:
model = nn.Sequential(
    nn.Linear(4, 8),  # 1 слой
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(8, 1)  # 2 слой
)

In [49]:
print(model)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=8, out_features=1, bias=True)
)


In [50]:
inputs = torch.ones(1, 4)
targets = torch.ones(1, 1)

## Прямой проход

In [51]:
# train
model.train()
with torch.no_grad():
    out1 = model[0](inputs)
    out2 = model[1](out1)
    out3 = model[2](out2)
    out_final_train = model[3](out3)

print("Режим обучения (Dropout включен):")
print("Выход 1-го слоя (до ReLU):", out1)
print("Выход после ReLU:", out2)
print("Выход после Dropout (некоторые выходы обнулятся):", out3)
print("Финальный выход сети:", out_final_train)

Режим обучения (Dropout включен):
Выход 1-го слоя (до ReLU): tensor([[ 1.5255,  0.1248,  0.4398,  0.9166,  0.4313,  0.2436, -1.0124, -0.5775]])
Выход после ReLU: tensor([[1.5255, 0.1248, 0.4398, 0.9166, 0.4313, 0.2436, 0.0000, 0.0000]])
Выход после Dropout (некоторые выходы обнулятся): tensor([[3.0509, 0.0000, 0.8796, 0.0000, 0.8626, 0.0000, 0.0000, 0.0000]])
Финальный выход сети: tensor([[0.7765]])


In [52]:
model.eval()
with torch.no_grad():
    out1_eval = model[0](inputs)
    out2_eval = model[1](out1_eval)
    out3_eval = model[2](out2_eval)
    out_final_eval = model[3](out3_eval)

print("\nРежим оценки (Dropout выключен):")
print("Выход 1-го слоя (до ReLU):", out1_eval)
print("Выход после ReLU:", out2_eval)
print("Выход после Dropout (значения должны сохраниться):", out3_eval)
print("Финальный выход сети:", out_final_eval)



Режим оценки (Dropout выключен):
Выход 1-го слоя (до ReLU): tensor([[ 1.5255,  0.1248,  0.4398,  0.9166,  0.4313,  0.2436, -1.0124, -0.5775]])
Выход после ReLU: tensor([[1.5255, 0.1248, 0.4398, 0.9166, 0.4313, 0.2436, 0.0000, 0.0000]])
Выход после Dropout (значения должны сохраниться): tensor([[1.5255, 0.1248, 0.4398, 0.9166, 0.4313, 0.2436, 0.0000, 0.0000]])
Финальный выход сети: tensor([[0.7178]])


## Обратный проход

In [53]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [54]:
def train_step(model, inputs, targets, criterion, optimizer):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f'{name}.grad до обновления:\n{param.grad}')
    optimizer.step()
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f'{name} после обновления:\n{param.data}')


In [55]:
print("\nОбучение с Dropout (режим обучения):")
model.train()  # Включаем Dropout
train_step(model, inputs, targets, criterion, optimizer)


Обучение с Dropout (режим обучения):
0.weight.grad до обновления:
tensor([[-0.0000, -0.0000, -0.0000, -0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.4104, -0.4104, -0.4104, -0.4104],
        [-0.0000, -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000, -0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])
0.bias.grad до обновления:
tensor([ 0.0000,  0.0000, -0.4104,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000])
3.weight.grad до обновления:
tensor([[ 0.0000,  0.0000, -1.2131,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])
3.bias.grad до обновления:
tensor([-1.3792])
0.weight после обновления:
tensor([[ 0.3823,  0.4150, -0.1171,  0.4593],
        [-0.1096,  0.1009, -0.2434,  0.2936],
        [ 0.4449, -0.3627,  0.4387,  0.0977],
        [ 0.3694,  0.0677,  0.2411, -0.0706],
        [ 0.3854,  0.0739, -0.2334,  0.1274],
        [-0.2304, -0.0586, -0.2031,  0.331

In [44]:
print("Обучение без Dropout (режим оценки):")
model.eval()  # Отключаем Dropout
train_step(model, inputs, targets, criterion, optimizer)


Обучение без Dropout (режим оценки):
0.weight.grad до обновления:
tensor([[-0.0494, -0.0494, -0.0494, -0.0494],
        [ 0.0498,  0.0498,  0.0498,  0.0498],
        [-0.0836, -0.0836, -0.0836, -0.0836],
        [-0.1640, -0.1640, -0.1640, -0.1640],
        [-0.1062, -0.1062, -0.1062, -0.1062],
        [ 0.0803,  0.0803,  0.0803,  0.0803],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])
0.bias.grad до обновления:
tensor([-0.0494,  0.0498, -0.0836, -0.1640, -0.1062,  0.0803,  0.0000,  0.0000])
3.weight.grad до обновления:
tensor([[-0.7927, -0.0648, -0.2392, -0.4763, -0.2241, -0.1266,  0.0000,  0.0000]])
3.bias.grad до обновления:
tensor([-0.5196])
0.weight после обновления:
tensor([[ 0.3828,  0.4155, -0.1166,  0.4598],
        [-0.1100,  0.1004, -0.2439,  0.2931],
        [ 0.4457, -0.3619,  0.4395,  0.0985],
        [ 0.3710,  0.0694,  0.2427, -0.0690],
        [ 0.3865,  0.0750, -0.2324,  0.1285],
        [-0.2312, -0.0594, -0.2039,  0.3309